# Taller Taxis NYC — EMR + PySpark

Viajes de los taxis amarillos de Nueva York en 2018: **102.871.387 viajes** en 12 archivos Parquet
(1,46 GB), leídos desde S3 por un clúster EMR de 1 primario + 2 core `m4.large`.

**Kernel: PySpark.** La sesión `spark` la crea el kernel, en el clúster.

| Decisión | Valor |
|---|---|
| Dato erróneo | recogida fuera de 2018 · duración ≤ 0 · duración mayor al tope (elegido con los percentiles) · zonas 264 y 265 |
| Qué se guarda (paso 12) | sólo 2018, con todas sus duraciones |

In [1]:
# Parámetros del taller
BUCKET = "taxis-nyc-936980625651"
BASE = f"s3://{BUCKET}"
TOPE_MIN = 180           # duración máxima válida, en minutos. Elegido con la referencia de Athena:
                         # el 99 % dura menos de 62 min; hasta 3 h la cantidad cae ~10× por tramo, y
                         # después deja de caer y aparece un pico de 218.865 viajes de 22 a 24 h
                         # (taxímetros sin cortar). Descarta el 0,23 %.
SIN_ZONA = [264, 265]    # 264 = Unknown · 265 = Outside of NYC: ninguna es un lugar

from pyspark import StorageLevel
from pyspark.sql import functions as F

# Las fechas del TLC son hora local de Nueva York sin zona horaria. Con la sesión en UTC,
# year(), month() y la resta trabajan sobre la hora tal como está escrita.
spark.conf.set("spark.sql.session.timeZone", "UTC")

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1791901354367_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## Paso 2 — Viajes y zonas, en dos DataFrames

Se leen **sólo las 4 columnas** que usa el taller, con un esquema explícito: tienen el mismo tipo en
los 12 meses. Leer la carpeta con su esquema completo puede fallar, porque `passenger_count` y
`RatecodeID` pasan de INT64 a DOUBLE en julio.

In [2]:
ESQUEMA_VIAJES = ("tpep_pickup_datetime timestamp, tpep_dropoff_datetime timestamp, "
                  "PULocationID long, DOLocationID long")
ESQUEMA_ZONAS = "LocationID int, Borough string, Zone string, service_zone string"

viajes = spark.read.schema(ESQUEMA_VIAJES).parquet(f"{BASE}/taxis/viajes/")
zonas = spark.read.csv(f"{BASE}/taxis/zonas/", header=True, schema=ESQUEMA_ZONAS)

print(f"viajes: {viajes.count():,}")   # control: 102.871.387
print(f"zonas:  {zonas.count():,}")    # control: 265
viajes.printSchema()
zonas.show(5, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

viajes: 102,871,387
zonas:  265
root
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)

+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
+----------+-------------+-----------------------+------------+
only showing top 5 rows

## Paso 3 — Lugar de recogida (`PU_ZONE`), hora de recogida, hora de dejada y `DOLocationID`

Zonas tiene 265 filas: `broadcast` la copia a cada nodo y los 102 millones de viajes no se mueven
por la red.

In [3]:
zonas_pu = zonas.select(F.col("LocationID").alias("PULocationID"),
                        F.col("Zone").alias("PU_ZONE"))

paso3 = (viajes.join(F.broadcast(zonas_pu), on="PULocationID")
               .select("PULocationID", "PU_ZONE",
                       F.col("tpep_pickup_datetime").alias("hora_recogida"),
                       F.col("tpep_dropoff_datetime").alias("hora_dejada"),
                       "DOLocationID"))

paso3.select("PU_ZONE", "hora_recogida", "hora_dejada", "DOLocationID").show(10, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------------------------+-------------------+-------------------+------------+
|PU_ZONE                       |hora_recogida      |hora_dejada        |DOLocationID|
+------------------------------+-------------------+-------------------+------------+
|Long Island City/Hunters Point|2018-03-01 00:01:34|2018-03-01 00:01:43|145         |
|Manhattan Valley              |2018-03-01 00:14:34|2018-03-01 00:28:13|244         |
|Upper West Side North         |2018-03-01 00:51:25|2018-03-01 00:59:54|152         |
|Long Island City/Hunters Point|2018-03-01 00:00:01|2018-03-01 00:00:17|145         |
|Long Island City/Hunters Point|2018-03-01 00:55:10|2018-03-01 00:56:36|145         |
|Long Island City/Hunters Point|2018-03-01 00:56:55|2018-03-01 00:57:01|145         |
|Queensbridge/Ravenswood       |2018-03-01 00:43:30|2018-03-01 00:50:40|193         |
|Queensbridge/Ravenswood       |2018-03-01 00:52:10|2018-03-01 00:52:21|193         |
|Astoria                       |2018-03-01 00:13:21|20

## Paso 4 — Segundo cruce: el nombre de la zona de destino

Los IDs quedan guardados para agrupar en los pasos 7 y 8: hay nombres de zona repetidos
("Corona" es la 56 y la 57), y agrupar por nombre sumaría zonas distintas.

In [4]:
zonas_do = zonas.select(F.col("LocationID").alias("DOLocationID"),
                        F.col("Zone").alias("DO_ZONE"))

paso4 = (paso3.join(F.broadcast(zonas_do), on="DOLocationID")
              .select("PULocationID", "DOLocationID", "PU_ZONE", "DO_ZONE",
                      F.col("hora_recogida").alias("PU_datetime"),
                      F.col("hora_dejada").alias("DO_datetime")))

paso4.select("PU_ZONE", "DO_ZONE", "PU_datetime", "DO_datetime").show(10, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------------------------+------------------------------+-------------------+-------------------+
|PU_ZONE                       |DO_ZONE                       |PU_datetime        |DO_datetime        |
+------------------------------+------------------------------+-------------------+-------------------+
|Long Island City/Hunters Point|Long Island City/Hunters Point|2018-03-01 00:01:34|2018-03-01 00:01:43|
|Manhattan Valley              |Washington Heights South      |2018-03-01 00:14:34|2018-03-01 00:28:13|
|Upper West Side North         |Manhattanville                |2018-03-01 00:51:25|2018-03-01 00:59:54|
|Long Island City/Hunters Point|Long Island City/Hunters Point|2018-03-01 00:00:01|2018-03-01 00:00:17|
|Long Island City/Hunters Point|Long Island City/Hunters Point|2018-03-01 00:55:10|2018-03-01 00:56:36|
|Long Island City/Hunters Point|Long Island City/Hunters Point|2018-03-01 00:56:55|2018-03-01 00:57:01|
|Queensbridge/Ravenswood       |Queensbridge/Ravenswood       |2

## Paso 5 — `trip_duration`, en minutos

Segundos Unix de la bajada menos los de la recogida, dividido 60. El resultado se guarda en
memoria (y en disco lo que no entre): todo lo que sigue sale de acá, y así no se relee S3 ni se
repiten los joins en cada consulta.

In [5]:
viajes_dur = paso4.withColumn(
    "trip_duration",
    (F.unix_timestamp("DO_datetime") - F.unix_timestamp("PU_datetime")) / 60)

viajes_dur.persist(StorageLevel.MEMORY_AND_DISK)
print(f"después de los dos joins: {viajes_dur.count():,}")   # control: igual que viajes

viajes_dur.select("PU_ZONE", "DO_ZONE", "PU_datetime", "DO_datetime",
                  F.round("trip_duration", 2).alias("trip_duration")).show(10, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

despu?s de los dos joins: 102,871,387
+------------------------------+------------------------------+-------------------+-------------------+-------------+
|PU_ZONE                       |DO_ZONE                       |PU_datetime        |DO_datetime        |trip_duration|
+------------------------------+------------------------------+-------------------+-------------------+-------------+
|Long Island City/Hunters Point|Long Island City/Hunters Point|2018-03-01 00:01:34|2018-03-01 00:01:43|0.15         |
|Manhattan Valley              |Washington Heights South      |2018-03-01 00:14:34|2018-03-01 00:28:13|13.65        |
|Upper West Side North         |Manhattanville                |2018-03-01 00:51:25|2018-03-01 00:59:54|8.48         |
|Long Island City/Hunters Point|Long Island City/Hunters Point|2018-03-01 00:00:01|2018-03-01 00:00:17|0.27         |
|Long Island City/Hunters Point|Long Island City/Hunters Point|2018-03-01 00:55:10|2018-03-01 00:56:36|1.43         |
|Long Island City/

## Antes de responder: qué tan sucios están los datos

Válido = recogida en 2018 y duración entre 0 y el tope. Los percentiles son los que justifican el
tope elegido.

In [6]:
es_2018 = F.year("PU_datetime") == 2018
dur = F.col("trip_duration")
valido = es_2018 & (dur > 0) & (dur <= TOPE_MIN)

viajes_dur.select(
    F.count("*").alias("total"),
    F.sum((~es_2018).cast("int")).alias("fuera_de_2018"),
    F.sum((es_2018 & (dur <= 0)).cast("int")).alias("duracion_no_positiva"),
    F.sum((es_2018 & (dur > TOPE_MIN)).cast("int")).alias("sobre_el_tope"),
    F.sum(valido.cast("int")).alias("validos"),
).show(truncate=False)

percentiles = viajes_dur.filter(es_2018 & (dur > 0)).approxQuantile(
    "trip_duration", [0.5, 0.9, 0.99, 0.999, 0.9999], 0.0001)
print("percentiles de duración (min) p50, p90, p99, p99.9, p99.99:", [round(p, 1) for p in percentiles])

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------+-------------+--------------------+-------------+---------+
|total    |fuera_de_2018|duracion_no_positiva|sobre_el_tope|validos  |
+---------+-------------+--------------------+-------------+---------+
|102871387|1282         |90375               |241065       |102538665|
+---------+-------------+--------------------+-------------+---------+

percentiles de duraci?n (min) p50, p90, p99, p99.9, p99.99: [11.2, 28.7, 62.4, 1426.0, 28837.0]

## Paso 6 — El viaje que más duró

Dos respuestas: el más largo **tal como viene** —que muestra el dato roto— y el más largo
**entre los válidos**.

In [7]:
columnas = ["PU_ZONE", "DO_ZONE", "PU_datetime", "DO_datetime",
            F.round("trip_duration", 1).alias("trip_duration"),
            F.round(F.col("trip_duration") / 60 / 24, 1).alias("dias")]

print("Tal como viene:")
viajes_dur.orderBy(F.desc("trip_duration")).select(*columnas).show(1, truncate=False)

print(f"Entre los válidos (2018, entre 0 y {TOPE_MIN:g} min):")
viajes_dur.filter(valido).orderBy(F.desc("trip_duration")).select(*columnas).show(1, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Tal como viene:
+--------------------+--------+-------------------+-------------------+-------------+----+
|PU_ZONE             |DO_ZONE |PU_datetime        |DO_datetime        |trip_duration|dias|
+--------------------+--------+-------------------+-------------------+-------------+----+
|TriBeCa/Civic Center|Kips Bay|2018-11-09 08:06:08|2018-11-29 08:43:06|28837.0      |20.0|
+--------------------+--------+-------------------+-------------------+-------------+----+
only showing top 1 row

Entre los v?lidos (2018, entre 0 y 180 min):
+-----------+------------+-------------------+-------------------+-------------+----+
|PU_ZONE    |DO_ZONE     |PU_datetime        |DO_datetime        |trip_duration|dias|
+-----------+------------+-------------------+-------------------+-------------+----+
|Murray Hill|Midtown East|2018-03-09 20:08:19|2018-03-09 23:08:19|180.0        |0.1 |
+-----------+------------+-------------------+-------------------+-------------+----+
only showing top 1 row

## Paso 7 — El destino más frecuente, sin datos erróneos ni las zonas 264 y 265

In [8]:
destinos = (viajes_dur
            .filter(valido & ~F.col("DOLocationID").isin(SIN_ZONA))
            .groupBy("DOLocationID", "DO_ZONE")
            .count()
            .orderBy(F.desc("count")))
destinos.show(5, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------+-------------------------+-------+
|DOLocationID|DO_ZONE                  |count  |
+------------+-------------------------+-------+
|236         |Upper East Side North    |3973487|
|161         |Midtown Center           |3895672|
|237         |Upper East Side South    |3716046|
|170         |Murray Hill              |3276379|
|230         |Times Sq/Theatre District|3171196|
+------------+-------------------------+-------+
only showing top 5 rows

## Paso 8 — El viaje más frecuente (origen-destino): promedio y desviación estándar

`stddev` es la desviación estándar **muestral**. Los valores absurdos la inflan: por eso va con el
filtro de validez.

In [9]:
pares = (viajes_dur
         .filter(valido
                 & ~F.col("PULocationID").isin(SIN_ZONA)
                 & ~F.col("DOLocationID").isin(SIN_ZONA))
         .groupBy("PULocationID", "PU_ZONE", "DOLocationID", "DO_ZONE")
         .agg(F.count("*").alias("viajes"),
              F.round(F.avg("trip_duration"), 2).alias("promedio_min"),
              F.round(F.stddev("trip_duration"), 2).alias("desviacion_estandar_min"))
         .orderBy(F.desc("viajes")))
pares.show(5, truncate=False)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------+---------------------+------------+---------------------+------+------------+-----------------------+
|PULocationID|PU_ZONE              |DOLocationID|DO_ZONE              |viajes|promedio_min|desviacion_estandar_min|
+------------+---------------------+------------+---------------------+------+------------+-----------------------+
|237         |Upper East Side South|236         |Upper East Side North|609664|6.28        |3.3                    |
|236         |Upper East Side North|236         |Upper East Side North|514950|4.38        |3.82                   |
|236         |Upper East Side North|237         |Upper East Side South|512198|7.92        |4.29                   |
|237         |Upper East Side South|237         |Upper East Side South|482256|5.03        |4.16                   |
|239         |Upper West Side South|238         |Upper West Side North|290409|4.85        |2.54                   |
+------------+---------------------+------------+---------------------+-

## Pasos 9 y 10 — Año y mes de la recogida, como enteros

In [10]:
viajes_ym = (viajes_dur
             .withColumn("year", F.year("PU_datetime").cast("int"))
             .withColumn("month", F.month("PU_datetime").cast("int")))
viajes_ym.printSchema()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- PU_ZONE: string (nullable = true)
 |-- DO_ZONE: string (nullable = true)
 |-- PU_datetime: timestamp (nullable = true)
 |-- DO_datetime: timestamp (nullable = true)
 |-- trip_duration: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

## Paso 11 — Meses de 2018, de más a menos viajes

Sólo 2018: si no, enero sumaría también los viajes de enero de 2001.

In [11]:
del_2018 = viajes_ym.filter(F.col("year") == 2018)
meses = del_2018.groupBy("month").count().orderBy(F.desc("count"))
meses.show(12)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-----+-------+
|month|  count|
+-----+-------+
|    3|9430359|
|    4|9305987|
|    5|9224816|
|   10|8834553|
|    1|8760903|
|    6|8714545|
|    2|8493198|
|   12|8194900|
|   11|8156023|
|    9|8048894|
|    8|7854947|
|    7|7850980|
+-----+-------+

## Paso 12 — Parquet particionado en `s3://bucket/viajes/year=…/month=…`

Se guarda 2018 con todas sus duraciones: 12 particiones, un archivo por mes.

In [12]:
salida = f"{BASE}/viajes/"
(del_2018
 .select("PU_ZONE", "DO_ZONE", "PU_datetime", "DO_datetime", "trip_duration", "year", "month")
 .repartition("year", "month")   # un archivo por mes: buen tamaño para Athena
 .write.mode("overwrite")
 .partitionBy("year", "month")
 .parquet(salida))

guardado = spark.read.parquet(salida)
print(f"guardadas: {guardado.count():,}")
guardado.groupBy("year", "month").count().orderBy("month").show(12)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

guardadas: 102,870,105
+----+-----+-------+
|year|month|  count|
+----+-----+-------+
|2018|    1|8760903|
|2018|    2|8493198|
|2018|    3|9430359|
|2018|    4|9305987|
|2018|    5|9224816|
|2018|    6|8714545|
|2018|    7|7850980|
|2018|    8|7854947|
|2018|    9|8048894|
|2018|   10|8834553|
|2018|   11|8156023|
|2018|   12|8194900|
+----+-----+-------+

## Pasos 13 y 14

- **13** — Crawler de Glue sobre `s3://taxis-nyc-936980625651/viajes/` → tabla `viajes` en la base
  `taxis`, consultada desde Athena (`infra/04-crawler.sh`).
- **14** — Este notebook, convertido en `spark/taller_taxis.py` y corrido como EMR Step
  (`infra/03-correr-paso-14.sh`). Tiene que dar los mismos números.